# Week 6-7: Multiple Weights & Learning Rate

## Recap From Last Week

In Weeks 4-5 we mastered gradient descent with **one weight**:
- Compute gradient: `∂L/∂w = 2(ŷ - y) × z`
- Update: `w = w - lr × gradient`

---

## This Week: The Real World Has Many Features

Real models don't have one feature — they have dozens or hundreds. A loan model uses age, income, credit score, debt ratio, employment years, ...

How does gradient descent handle multiple weights at once?

**The answer is elegant:** Each weight gets its own gradient, computed independently using the **same error** but the weight's **own feature value**.

$$\frac{\partial L}{\partial w_j} = 2(\hat{y} - y) \times z_j$$

Then every weight updates simultaneously:

$$w_j^{\text{new}} = w_j - lr \times \frac{\partial L}{\partial w_j}$$

**Key word: simultaneously.** We compute ALL gradients with the old weights first, THEN update everything. This is crucial — we will see what goes wrong if you don't.

---

## Two Topics This Week

1. **Multiple weights** — Independent gradients, simultaneous updates, feature scale effects
2. **Learning rate** — Too small, just right, too large, catastrophically large


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

plt.rcParams.update({
    'figure.facecolor': '#f9f9f9',
    'axes.facecolor':   '#f9f9f9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        11,
})

np.random.seed(42)
print("Libraries loaded. Let's explore multiple weights!")

---
# Model A: Two Features — Loan Approval

## Model Architecture

$$\hat{y} = w_{\text{age}} \times z_{\text{age}} + w_{\text{income}} \times z_{\text{income}} + b$$

**Initial weights:** `w_age = 0.2`, `w_income = 0.3`, `b = 0.0`  
**Learning rate:** `lr = 0.1`

We will process **3 different applicants** and observe how each produces different gradients for the same pair of weights.

**The important insight:** `z_age = 1.5` vs `z_income = 0.5` — the age feature has a larger value, so it will produce a **larger gradient magnitude** for `w_age` than for `w_income`, even though the error `(ŷ - y)` is the same for both.

In [ ]:
# ─── Model A: Two-feature loan approval ──────────────────────────────────────
# 3 training examples: [z_age, z_income, y_true]
data_A = [
    {'name': 'Applicant 1', 'z_age': 1.5, 'z_income': 0.5,  'y': 1.0},  # approved, older, lower income
    {'name': 'Applicant 2', 'z_age': -0.8,'z_income': 1.2,  'y': 0.0},  # rejected, young, high income but risky
    {'name': 'Applicant 3', 'z_age': 0.3, 'z_income': -0.6, 'y': 1.0},  # approved, average age, low income but safe
]

w_age    = 0.2
w_income = 0.3
b        = 0.0
lr_A     = 0.1

print("=" * 72)
print("MODEL A: LOAN APPROVAL — Two Features")
print("  Initial: w_age=0.2  w_income=0.3  b=0.0  lr=0.1")
print("=" * 72)

for ex in data_A:
    y_hat = w_age * ex['z_age'] + w_income * ex['z_income'] + b
    error = y_hat - ex['y']
    loss  = error ** 2

    grad_age    = 2 * error * ex['z_age']
    grad_income = 2 * error * ex['z_income']
    grad_b      = 2 * error

    w_age_new    = w_age    - lr_A * grad_age
    w_income_new = w_income - lr_A * grad_income
    b_new        = b        - lr_A * grad_b

    print(f"\n  {ex['name']}: z_age={ex['z_age']:+.1f}, z_income={ex['z_income']:+.1f}, y={ex['y']}")
    print(f"    ŷ = {w_age:.2f}×{ex['z_age']:+.1f} + {w_income:.2f}×{ex['z_income']:+.1f} + {b:.2f} = {y_hat:.4f}")
    print(f"    Error = ŷ - y = {error:+.4f}")
    print(f"    ∂L/∂w_age    = 2 × {error:+.4f} × {ex['z_age']:+.1f} = {grad_age:+.4f}")
    print(f"    ∂L/∂w_income = 2 × {error:+.4f} × {ex['z_income']:+.1f} = {grad_income:+.4f}")
    print(f"    w_age:    {w_age:.3f} → {w_age_new:.4f}")
    print(f"    w_income: {w_income:.3f} → {w_income_new:.4f}")
    print(f"    NOTE: |grad_age|={abs(grad_age):.4f}  vs  |grad_income|={abs(grad_income):.4f}"
          f"  — z_age larger, so bigger gradient")

**Key observation:** The gradient for `w_age` is consistently larger in magnitude than the gradient for `w_income` — because `z_age` is larger. This means the model learns the age weight faster. In Week 8 we will see how normalization keeps this under control.

---
# Model B: Three Features — Student Grades

## Model Architecture

$$\hat{y} = w_{\text{study}} \times z_{\text{study}} + w_{\text{attend}} \times z_{\text{attend}} + w_{\text{sleep}} \times z_{\text{sleep}} + b$$

We will process one student who **studied a lot** but still underperformed. The model should learn that study hours matter most.

**Notice:** All three gradients share the **same error** `(ŷ - y)`. What differs is the feature value `z_j` for each weight. The weight that corresponds to the highest-`z` feature gets the biggest gradient.

In [ ]:
# ─── Model B: Student grades, three features ─────────────────────────────────
w_study  = 0.1
w_attend = 0.2
w_sleep  = 0.15
b_B      = 0.0
lr_B     = 0.1

# One student: studied intensely, average attendance, below-average sleep
z_study  =  2.1    # very high study hours
z_attend =  0.4    # average attendance
z_sleep  = -0.7    # below average sleep
y_B      =  1.0    # passed with flying colours

y_hat_B  = w_study * z_study + w_attend * z_attend + w_sleep * z_sleep + b_B
error_B  = y_hat_B - y_B
loss_B   = error_B ** 2

grad_study  = 2 * error_B * z_study
grad_attend = 2 * error_B * z_attend
grad_sleep  = 2 * error_B * z_sleep
grad_b_B    = 2 * error_B

print("=" * 60)
print("MODEL B: STUDENT GRADES — Three Features")
print("=" * 60)
print(f"  Features:  z_study={z_study:+.1f},  z_attend={z_attend:+.1f},  z_sleep={z_sleep:+.1f}")
print(f"  Weights:   w_study={w_study},  w_attend={w_attend},  w_sleep={w_sleep}")
print()
print(f"  Prediction: ŷ = {w_study}×{z_study} + {w_attend}×{z_attend} + {w_sleep}×{z_sleep}")
print(f"                = {w_study*z_study:.3f} + {w_attend*z_attend:.3f} + {w_sleep*z_sleep:.3f}")
print(f"                = {y_hat_B:.4f}")
print(f"  True label:   y = {y_B}")
print(f"  Error = ŷ - y = {error_B:.4f}  (negative — underpredicting)")
print()
print(f"  Gradients (same error, different z values):")
print(f"    ∂L/∂w_study  = 2 × {error_B:.4f} × {z_study:+.1f} = {grad_study:+.5f}  ← LARGEST")
print(f"    ∂L/∂w_attend = 2 × {error_B:.4f} × {z_attend:+.1f} = {grad_attend:+.5f}")
print(f"    ∂L/∂w_sleep  = 2 × {error_B:.4f} × {z_sleep:+.1f} = {grad_sleep:+.5f}")
print(f"    ∂L/∂b        = 2 × {error_B:.4f}         = {grad_b_B:+.5f}")
print()
print(f"  Updates (lr={lr_B}):")
print(f"    w_study:  {w_study:.3f} → {w_study - lr_B*grad_study:.4f}  (step={-lr_B*grad_study:+.4f})")
print(f"    w_attend: {w_attend:.3f} → {w_attend - lr_B*grad_attend:.4f}  (step={-lr_B*grad_attend:+.4f})")
print(f"    w_sleep:  {w_sleep:.3f} → {w_sleep - lr_B*grad_sleep:.4f}  (step={-lr_B*grad_sleep:+.4f})")
print()
print("  Key lesson: High study hours (z=2.1) drives the biggest gradient.")
print("  The model learns 'study hours' fastest because it has the highest feature value.")

In [ ]:
# ─── Bar chart: gradient magnitude per feature ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model B: Gradient Magnitude Per Feature\n(Same error drives all gradients — but z value scales them)', fontsize=12, fontweight='bold')

features      = ['w_study\n(z=+2.1)', 'w_attend\n(z=+0.4)', 'w_sleep\n(z=-0.7)', 'bias\n(z=1)']
grad_vals     = [grad_study, grad_attend, grad_sleep, grad_b_B]
z_vals_disp   = [z_study, z_attend, z_sleep, 1.0]
colors        = ['#d62728' if g < 0 else '#1f77b4' for g in grad_vals]

# Left: gradient values
bars = axes[0].bar(features, grad_vals, color=colors, edgecolor='black', linewidth=0.7)
axes[0].axhline(0, color='black', lw=1)
axes[0].set_ylabel('Gradient value')
axes[0].set_title('Gradient for Each Weight')
for bar, val in zip(bars, grad_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + (0.01 if val >= 0 else -0.04),
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9)

# Right: z-value (feature scale) vs gradient magnitude
axes[1].scatter(np.abs(z_vals_disp), np.abs(grad_vals), s=120, zorder=5,
                color=['#1f77b4','#ff7f0e','#2ca02c','#9467bd'])
for i, (zv, gv, fname) in enumerate(zip(z_vals_disp, grad_vals, features)):
    axes[1].annotate(fname.split('\n')[0], (abs(zv), abs(gv)),
                     textcoords='offset points', xytext=(6, 2), fontsize=9)
axes[1].set_xlabel('|z| (feature magnitude)')
axes[1].set_ylabel('|gradient| (gradient magnitude)')
axes[1].set_title('Feature Scale → Gradient Scale\n(linear relationship)')

# Fit a line to show linear relationship
z_abs = np.array([abs(z) for z in z_vals_disp])
g_abs = np.array([abs(g) for g in grad_vals])
slope = np.polyfit(z_abs, g_abs, 1)
x_line = np.linspace(0, max(z_abs) + 0.2, 50)
axes[1].plot(x_line, np.polyval(slope, x_line), 'k--', lw=1.5, alpha=0.6, label='Linear fit')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
# Model C: Mini-Dataset — House Price (5 Examples)

So far we have updated weights one example at a time. In real training we typically average gradients over a **batch** of examples.

$$\overline{\nabla} w_j = \frac{1}{N} \sum_{i=1}^N 2(\hat{y}_i - y_i) \times z_{ij}$$

This reduces noise and gives more stable updates. Let's run 10 iterations on 5 house price examples.

In [ ]:
# ─── Model C: House price mini-dataset ───────────────────────────────────────
# 5 training examples: [z_size, z_rooms, y_price]
# All normalized to similar scale
Z_house = np.array([
    [1.2,  0.8],
    [0.3, -0.2],
    [-1.1, -0.9],
    [0.7,  1.5],
    [-0.5,  0.4],
])  # shape (5, 2)

y_house = np.array([1.0, 0.4, -0.8, 0.9, -0.1])   # normalized prices

w_C  = np.array([0.0, 0.0])   # [w_size, w_rooms]
b_C  = 0.0
lr_C = 0.1

n_iter_C = 10
history_C = []

print("=" * 65)
print("MODEL C: HOUSE PRICE — 5 Training Examples, 10 Iterations")
print("=" * 65)
print(f"  Features: z_size, z_rooms | Weights: {w_C} | lr={lr_C}")
print()
print(f"{'Iter':>5} {'w_size':>10} {'w_rooms':>10} {'bias':>8} {'Avg Loss':>10}")
print("-" * 50)

for it in range(n_iter_C):
    y_hat_C = Z_house @ w_C + b_C          # predictions, shape (5,)
    errors  = y_hat_C - y_house            # errors, shape (5,)
    loss    = np.mean(errors ** 2)         # mean squared error

    # Gradients (averaged over all 5 examples)
    grad_w_C = 2 * (Z_house.T @ errors) / len(y_house)   # shape (2,)
    grad_b_C = 2 * np.mean(errors)

    history_C.append({'iter': it, 'w': w_C.copy(), 'b': b_C, 'loss': loss})

    if it % 2 == 0 or it == n_iter_C - 1:
        print(f"  {it:>3}   {w_C[0]:>10.5f} {w_C[1]:>10.5f} {b_C:>8.5f} {loss:>10.5f}")

    # Simultaneous update
    w_C = w_C - lr_C * grad_w_C
    b_C = b_C - lr_C * grad_b_C

print()
print(f"  Final weights: w_size={w_C[0]:.5f}, w_rooms={w_C[1]:.5f}, b={b_C:.5f}")
print(f"  Final loss: {history_C[-1]['loss']:.5f}")

In [ ]:
# ─── Visualize weight trajectory and loss for Model C ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Model C: House Price — Weight Trajectories over 10 Iterations', fontsize=12, fontweight='bold')

iters_C   = [h['iter'] for h in history_C]
w_size_h  = [h['w'][0] for h in history_C]
w_rooms_h = [h['w'][1] for h in history_C]
losses_C  = [h['loss']  for h in history_C]

# Loss
axes[0].plot(iters_C, losses_C, 'o-', color='crimson', lw=2, ms=6)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Mean Squared Loss')
axes[0].set_title('Loss Decreasing')
axes[0].fill_between(iters_C, losses_C, alpha=0.15, color='crimson')

# Individual weights
axes[1].plot(iters_C, w_size_h,  'o-', color='steelblue', lw=2, ms=6, label='w_size')
axes[1].plot(iters_C, w_rooms_h, 's-', color='darkorange', lw=2, ms=6, label='w_rooms')
axes[1].axhline(0, color='gray', lw=1, linestyle='--', alpha=0.5)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Weight value')
axes[1].set_title('Weights Converging')
axes[1].legend(fontsize=9)

# 2D weight trajectory (contour-style path)
axes[2].plot(w_size_h, w_rooms_h, 'o-', color='purple', lw=2, ms=6)
axes[2].scatter([w_size_h[0]], [w_rooms_h[0]], color='red', s=100, zorder=6, label='Start (0,0)')
axes[2].scatter([w_size_h[-1]], [w_rooms_h[-1]], color='green', s=100, zorder=6, marker='*', label='End')
for i in range(0, len(iters_C), 2):
    axes[2].annotate(f'it{iters_C[i]}', (w_size_h[i], w_rooms_h[i]),
                     textcoords='offset points', xytext=(4, 3), fontsize=8)
axes[2].set_xlabel('w_size')
axes[2].set_ylabel('w_rooms')
axes[2].set_title('2D Weight Space Path')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
# Simultaneous vs Sequential Updates — A Critical Mistake to Avoid

## The Wrong Way (Sequential)

Imagine you update `w1` first, then use the **new** `w1` to compute `w2`'s gradient:

```
# WRONG:
grad_w1 = 2 * error * z1
w1 = w1 - lr * grad_w1          # w1 is updated here
y_hat_new = w1 * z1 + w2 * z2   # uses new w1!
error_new = y_hat_new - y
grad_w2 = 2 * error_new * z2    # different error!
w2 = w2 - lr * grad_w2
```

## The Right Way (Simultaneous)

```
# CORRECT:
y_hat = w1 * z1 + w2 * z2
error = y_hat - y
grad_w1 = 2 * error * z1        # both computed with OLD weights
grad_w2 = 2 * error * z2
w1 = w1 - lr * grad_w1          # now update both simultaneously
w2 = w2 - lr * grad_w2
```

Let's see the numerical difference:

In [ ]:
# ─── Simultaneous vs sequential update demonstration ─────────────────────────
w1_init, w2_init = 0.3, 0.5
z1, z2 = 1.5, 0.5
y_ex   = 1.0
lr_ex  = 0.2

# ─── SIMULTANEOUS (correct) ───
y_hat_sim = w1_init * z1 + w2_init * z2
error_sim = y_hat_sim - y_ex
grad_w1_sim = 2 * error_sim * z1
grad_w2_sim = 2 * error_sim * z2
w1_sim = w1_init - lr_ex * grad_w1_sim
w2_sim = w2_init - lr_ex * grad_w2_sim

# ─── SEQUENTIAL (wrong) ───
y_hat_seq   = w1_init * z1 + w2_init * z2
error_seq   = y_hat_seq - y_ex
grad_w1_seq = 2 * error_seq * z1
w1_seq      = w1_init - lr_ex * grad_w1_seq   # update w1 first!

y_hat_seq2  = w1_seq * z1 + w2_init * z2      # now use NEW w1
error_seq2  = y_hat_seq2 - y_ex
grad_w2_seq = 2 * error_seq2 * z2             # different error!
w2_seq      = w2_init - lr_ex * grad_w2_seq

print("=" * 62)
print("SIMULTANEOUS vs SEQUENTIAL WEIGHT UPDATES")
print("=" * 62)
print(f"  Setup: w1={w1_init}, w2={w2_init}, z1={z1}, z2={z2}, y={y_ex}, lr={lr_ex}")
print()
print("  CORRECT (simultaneous):")
print(f"    Shared error  = {error_sim:.4f}")
print(f"    grad_w1 = 2 × {error_sim:.4f} × {z1} = {grad_w1_sim:.4f}")
print(f"    grad_w2 = 2 × {error_sim:.4f} × {z2} = {grad_w2_sim:.4f}")
print(f"    w1_new = {w1_init} - {lr_ex}×{grad_w1_sim:.4f} = {w1_sim:.5f}")
print(f"    w2_new = {w2_init} - {lr_ex}×{grad_w2_sim:.4f} = {w2_sim:.5f}")
print()
print("  WRONG (sequential — w1 updated before w2's gradient):")
print(f"    grad_w1 computed with error={error_seq:.4f} → w1 updated to {w1_seq:.5f}")
print(f"    Now y_hat recomputed with NEW w1: {y_hat_seq2:.5f}")
print(f"    New error for w2 = {error_seq2:.5f}  (different from {error_seq:.4f}!)")
print(f"    grad_w2 = 2 × {error_seq2:.5f} × {z2} = {grad_w2_seq:.5f}")
print(f"    w2_new = {w2_init} - {lr_ex}×{grad_w2_seq:.5f} = {w2_seq:.5f}")
print()
print(f"  DIFFERENCE:")
print(f"    w1: same result ({w1_sim:.5f} vs {w1_seq:.5f})")
print(f"    w2: {w2_sim:.5f} (correct) vs {w2_seq:.5f} (wrong) — diff={abs(w2_sim-w2_seq):.6f}")
print()
print("  This diverges more when lr is large or weights are interdependent.")

---
# Learning Rate: Four Experiments

The learning rate `lr` controls how large a step we take each iteration. It is arguably the **most important hyperparameter** to get right.

| Learning Rate | Behaviour |
|---|---|
| `lr = 0.001` | Very small steps. Converges but very slowly. |
| `lr = 0.1` | Good balance. Converges in a reasonable number of steps. |
| `lr = 0.5` | Large steps. May oscillate around the minimum but usually converges. |
| `lr = 2.0` | Way too large. Weight overshoots and diverges to infinity. |

We will use a simple 1D problem: single weight, single feature, one training example.

In [ ]:
# ─── Learning rate experiments ────────────────────────────────────────────────
z_lr   = 1.0
y_lr   = 1.0
w_init = 0.0
n_iter_lr = 50

learning_rates = {
    'lr=0.001 (too small)': 0.001,
    'lr=0.1   (good)':      0.1,
    'lr=0.5   (too large)': 0.5,
    'lr=2.0   (diverges)':  2.0,
}

colors_lr = ['#aec7e8', '#1f77b4', '#ff7f0e', '#d62728']
results_lr = {}

for (label, lr_val), color in zip(learning_rates.items(), colors_lr):
    w   = w_init
    lrs = []
    for _ in range(n_iter_lr):
        y_hat = w * z_lr
        error = y_hat - y_lr
        loss  = error ** 2
        grad  = 2 * error * z_lr
        lrs.append(min(loss, 1e6))   # clamp for display
        w = w - lr_val * grad
    results_lr[label] = {'losses': lrs, 'color': color, 'lr': lr_val}

# Print summary
print(f"{'Learning Rate':<28} {'Loss at iter 5':>15} {'Loss at iter 20':>16} {'Loss at iter 50':>16}")
print("-" * 78)
for label, res in results_lr.items():
    l5  = res['losses'][4]
    l20 = res['losses'][19]
    l50 = res['losses'][49]
    print(f"  {label:<26} {l5:>15.5f} {l20:>16.5f} {l50:>16.5f}")

In [ ]:
# ─── Learning rate comparison chart ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Learning Rate Comparison — Loss vs Iteration', fontsize=13, fontweight='bold')

iters_lr = range(n_iter_lr)

for label, res in results_lr.items():
    axes[0].plot(iters_lr, res['losses'], lw=2, color=res['color'], label=label)
    axes[1].semilogy(iters_lr, [max(l, 1e-10) for l in res['losses']],
                     lw=2, color=res['color'], label=label)

axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss')
axes[0].set_title('Linear Scale\n(diverging lr=2.0 dominates the plot)')
axes[0].legend(fontsize=9)
axes[0].set_ylim(-0.05, 2.5)

axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Log Scale\n(easier to compare convergence rates)')
axes[1].legend(fontsize=9)
axes[1].axhline(1e-4, color='gray', lw=1, linestyle=':', alpha=0.7, label='Convergence threshold')

plt.tight_layout()
plt.show()

**Reading the log-scale chart:**

- **`lr=0.001` (light blue):** Barely moves in 50 iterations. After 50 steps, loss is still far from zero. In a real problem with hundreds of weights, this would need thousands of epochs.

- **`lr=0.1` (dark blue):** Rapid decrease, smooth convergence. Reaches near-zero loss around iteration 20.

- **`lr=0.5` (orange):** Faster at first but oscillates. Converges but noisily. Notice the zig-zag pattern as it bounces around the minimum.

- **`lr=2.0` (red):** Loss explodes upward. The weight overshoots the minimum so badly each step that it gets further away, not closer. This is **divergence**.

In [ ]:
# ─── Visualise weight trajectory for each lr on the loss landscape ───────────
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Weight Trajectories on Error Landscape for Different Learning Rates', fontsize=13, fontweight='bold')

lr_configs = [
    (0.001, 'lr=0.001 (too small)', '#aec7e8'),
    (0.1,   'lr=0.1   (good)',      '#1f77b4'),
    (0.5,   'lr=0.5   (oscillates)','#ff7f0e'),
    (2.0,   'lr=2.0   (diverges)',  '#d62728'),
]

w_opt_lr = y_lr / z_lr   # = 1.0

for ax, (lr_val, label, color) in zip(axes.flatten(), lr_configs):
    w  = w_init
    ws = [w]
    n_steps = 15 if lr_val < 1.0 else 8
    for _ in range(n_steps - 1):
        y_h = w * z_lr
        g   = 2 * (y_h - y_lr) * z_lr
        w   = w - lr_val * g
        ws.append(np.clip(w, -3, 3))

    # Plot landscape
    lo_w = min(min(ws), w_opt_lr) - 0.5
    hi_w = max(max(ws), w_opt_lr) + 0.5
    ww   = np.linspace(lo_w, hi_w, 300)
    ll   = (ww * z_lr - y_lr) ** 2
    ax.plot(ww, ll, color='gray', lw=2, alpha=0.6, label='Loss landscape')

    # Plot trajectory
    losses_traj = [(wi * z_lr - y_lr) ** 2 for wi in ws]
    ax.plot(ws, losses_traj, 'o-', color=color, lw=2, ms=7, label='Path')
    ax.scatter([ws[0]],  [losses_traj[0]],  color='red',   zorder=6, s=100, label='Start')
    ax.scatter([ws[-1]], [losses_traj[-1]], color='black', zorder=6, s=100, marker='D', label='End')
    ax.scatter([w_opt_lr], [0], color='green', s=120, marker='*', zorder=6, label=f'Optimal w={w_opt_lr}')

    ax.set_xlabel('Weight w')
    ax.set_ylabel('Loss')
    ax.set_title(label, fontsize=11, fontweight='bold', color=color)
    ax.legend(fontsize=8)
    ax.set_ylim(-0.05, max(losses_traj) * 1.15 + 0.05)

plt.tight_layout()
plt.show()

---
# 2D Weight Update Visualization (Contour Plot)

With two weights, the loss landscape becomes a **bowl shape** in 2D (like the inside of a satellite dish). Gradient descent navigates across this bowl toward the lowest point.

We can visualize this with a **contour plot** where darker shading means lower loss.

In [ ]:
# ─── 2D Contour plot with gradient descent arrows ─────────────────────────────
# Single training example: z = [1.5, 0.5], y = 1.0
z1_c, z2_c = 1.5, 0.5
y_c        = 1.0

# Loss as a function of w1, w2: L = (w1*z1 + w2*z2 - y)^2
w1_vals = np.linspace(-0.5, 1.2, 200)
w2_vals = np.linspace(-0.5, 2.5, 200)
W1, W2  = np.meshgrid(w1_vals, w2_vals)
LOSS    = (W1 * z1_c + W2 * z2_c - y_c) ** 2

# Run gradient descent on this 2D problem
w1_gd, w2_gd = 0.0, 0.0
lr_2d = 0.1
path_w1, path_w2, path_loss = [w1_gd], [w2_gd], [(w1_gd*z1_c + w2_gd*z2_c - y_c)**2]

for _ in range(18):
    y_hat_2d = w1_gd * z1_c + w2_gd * z2_c
    err_2d   = y_hat_2d - y_c
    g1       = 2 * err_2d * z1_c
    g2       = 2 * err_2d * z2_c
    w1_gd   -= lr_2d * g1
    w2_gd   -= lr_2d * g2
    path_w1.append(w1_gd)
    path_w2.append(w2_gd)
    path_loss.append((w1_gd*z1_c + w2_gd*z2_c - y_c)**2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Gradient Descent in 2D Weight Space', fontsize=13, fontweight='bold')

# Contour plot
contour = axes[0].contourf(W1, W2, LOSS, levels=30, cmap='Blues_r', alpha=0.85)
plt.colorbar(contour, ax=axes[0], label='Loss')
axes[0].contour(W1, W2, LOSS, levels=10, colors='white', alpha=0.3, linewidths=0.7)

# Gradient descent path with arrows
axes[0].plot(path_w1, path_w2, 'o-', color='crimson', lw=2, ms=5, zorder=5, label='GD path')
for i in range(len(path_w1) - 1):
    dx = path_w1[i+1] - path_w1[i]
    dy = path_w2[i+1] - path_w2[i]
    if i % 3 == 0:
        axes[0].annotate('', xy=(path_w1[i+1], path_w2[i+1]),
                         xytext=(path_w1[i], path_w2[i]),
                         arrowprops=dict(arrowstyle='->', color='crimson', lw=1.5))

axes[0].scatter([path_w1[0]], [path_w2[0]], color='yellow', s=120, zorder=7, edgecolors='black', label='Start')
axes[0].scatter([path_w1[-1]], [path_w2[-1]], color='lime', s=120, zorder=7, marker='*', edgecolors='black', label='End')
axes[0].set_xlabel('w1 (weight for z1)')
axes[0].set_ylabel('w2 (weight for z2)')
axes[0].set_title('Loss Landscape + Path\n(darker = lower loss)')
axes[0].legend(fontsize=9)

# Loss over iterations
axes[1].plot(range(len(path_loss)), path_loss, 'o-', color='steelblue', lw=2, ms=6)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Curve for the 2D Path')
axes[1].fill_between(range(len(path_loss)), path_loss, alpha=0.15, color='steelblue')

plt.tight_layout()
plt.show()

**Reading the 2D contour plot:**

- Each **ring** is a contour line — all points on a ring have the same loss
- **Darker blue** means lower loss (we want to be in the dark center)
- The **yellow dot** is where we start (high loss, outer ring)
- The **green star** is where we end up (low loss, near center)
- The path curves because the loss bowl is elongated — the gradient points slightly off-center toward the optimal

Notice the path is not a straight line — it zig-zags a bit because the bowl is not perfectly circular. This is typical in models with multiple features at different scales.

---
# Summary & Key Takeaways

## Multiple Weights

| Rule | Detail |
|---|---|
| **Same error for all gradients** | `(ŷ - y)` is computed once, shared across all weight gradients |
| **Feature scales the gradient** | `∂L/∂wⱼ = 2(ŷ-y)×zⱼ` — large `zⱼ` → large gradient |
| **Simultaneous updates** | Compute ALL gradients first with old weights, THEN update all weights |
| **Averaged gradients** | When processing a batch, average gradients over examples |

## Learning Rate

| Value | Effect |
|---|---|
| Too small (`0.001`) | Converges very slowly — takes many epochs |
| Just right (`0.01`–`0.1`) | Smooth convergence in reasonable time |
| Too large (`0.5`) | Oscillates but may converge — watch the log plot |
| Way too large (`≥ 2.0`) | Diverges — loss goes to infinity |

**Practical advice:** Start with `lr = 0.01` or `lr = 0.1`. Look at the loss curve — if it's flat, increase. If it explodes, decrease.

## Next Week

In Week 8 we will put everything together into a **complete training loop** — generating synthetic datasets, normalizing features, running multiple epochs, monitoring convergence, and evaluating final performance.